# Multiproduct MLP benchmark

Protocol aligned with ICDN:
- Optuna on 3 expanding folds × 3 seeds.
- Final k-fold: 5 expanding folds × 5 seeds
- Block bootstrap on the 80/20 holdout

One-phase Huber MLP. Elasticities = Jacobian. No Phase 0/1, no pair dataset.

In [1]:
import sys
import json
import random
from pathlib import Path
from dataclasses import replace

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import torch
import optuna

from src.dominick import DominickDataLoader
from src.dominick.multiproduct_builder import MultiProductBuilder
from src.nn.data import ColumnEncoder, DataLoaderFactory
from src.multiproduct import MultiProductDataset
from src.utils import TemporalSplitter, BlockBootstrapSampler
from src.benchmarks import MLPConfig, DemandMLPPipeline

/home/thebigmonster/Github/nn-elasticity/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
BASE_SEED = 42

N_UPCS = 5
MIN_TRAIN_FRAC = 0.5
TRAIN_FRAC = 0.8
BLOCK_SIZE = 4

PROTOCOL = "nested_temporal"
N_INNER_FOLDS = 3
TUNE_SEEDS = [11, 29, 42]
N_TRIALS_NESTED = 20
N_TRIALS_HOLDOUT = 20

N_OUTER_FOLDS = 5
EVAL_SEEDS = [11, 29, 42, 77, 123]
N_BOOTSTRAP = 20

HIDDEN_OPTIONS = {
    "64_32":      (64, 32),
    "128_64":     (128, 64),
    "192_96":     (192, 96),
    "256_128":    (256, 128),
    "256_128_64": (256, 128, 64),
}

DATA_DIR = Path("../data")
RESULTS_DIR = Path("../results")
NESTED_DIR = RESULTS_DIR / "nested"
DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
NESTED_DIR.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cuda


In [3]:
def set_all_seeds(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_all_seeds(BASE_SEED)

# Data

In [4]:
loader = DominickDataLoader()
df = loader.load("elasticity_dataset.csv")

encoder = ColumnEncoder()
_, store_cats = encoder.factorize(df, "store_code", sort=True)
_, week_cats  = encoder.factorize(df, "week_id", sort=True)
_, brand_cats = encoder.factorize(df, "brand_family_norm", sort=True)
_, style_cats = encoder.factorize(df, "style_segment_norm", sort=True)

n_stores = len(store_cats)
n_brands = len(brand_cats)
n_styles = len(style_cats)

brand_map = {v: i + 1 for i, v in enumerate(brand_cats)}
style_map = {v: i + 1 for i, v in enumerate(style_cats)}
df["brand_family_norm"] = df["brand_family_norm"].map(brand_map).fillna(0).astype(int)
df["style_segment_norm"] = df["style_segment_norm"].map(style_map).fillna(0).astype(int)

mp_builder = MultiProductBuilder()
sorted_weeks = sorted(df["week_id"].unique())
n_sample = max(1, int(len(sorted_weeks) * MIN_TRAIN_FRAC))
sample_weeks = sorted_weeks[:n_sample]
mp_builder.fit_panel(df[df["week_id"].isin(sample_weeks)], n_upcs=N_UPCS)
n_upcs = mp_builder.n
upc_names = mp_builder.selected_upcs
n_product_feats = len(MultiProductDataset.PER_PRODUCT_COLS) + 1

store_map = {v: i for i, v in enumerate(store_cats)}
week_map = {v: i for i, v in enumerate(week_cats)}

print(f"Stores: {n_stores}  weeks: {len(week_cats)}  UPCs: {list(upc_names)}")

Stores: 70  weeks: 302  UPCs: [3410057306, 3410017306, 3410010505, 3410015306, 1820011168]


# Temporal folds

In [5]:
splitter = TemporalSplitter(week_col="week_id")

nested_plans = splitter.nested_expanding_splits(
    df=df,
    n_outer=N_OUTER_FOLDS,
    n_inner=N_INNER_FOLDS,
    min_train_frac=MIN_TRAIN_FRAC,
)

def materialize(plan_or_pair):
    if isinstance(plan_or_pair, dict):
        tr_w, va_w = mp_builder.make_fold_frames(plan_or_pair["outer_train"], plan_or_pair["outer_val"])
        plan_or_pair["outer_train"], plan_or_pair["outer_val"] = tr_w, va_w
        plan_or_pair["inner_splits"] = [
            mp_builder.make_fold_frames(tr, va) for tr, va in plan_or_pair["inner_splits"]
        ]
        return plan_or_pair
    tr, va = plan_or_pair
    return mp_builder.make_fold_frames(tr, va)

nested_plans = [materialize(p) for p in nested_plans]

train_final_long, val_final_long = splitter.single_split(df, train_frac=TRAIN_FRAC)
holdout_inner_long = splitter.expanding_splits(
    train_final_long, n_folds=N_INNER_FOLDS, min_train_frac=MIN_TRAIN_FRAC,
)
holdout_weeks = set(val_final_long["week_id"].unique())
for _, inner_val in holdout_inner_long:
    leak = set(inner_val["week_id"].unique()) & holdout_weeks
    if leak:
        raise RuntimeError(f"Holdout leak: {sorted(leak)[:10]}")

holdout_inner = [mp_builder.make_fold_frames(tr, va) for tr, va in holdout_inner_long]
train_final, val_final = mp_builder.make_fold_frames(train_final_long, val_final_long)
train_weeks_final = sorted(train_final["week_id"].unique())

# Helpers

In [6]:
def encode_fold(train_wide, val_wide):
    train_wide, val_wide = train_wide.copy(), val_wide.copy()
    for w in (train_wide, val_wide):
        w["store_code"] = w["store_code"].map(store_map)
        w["week_id"] = w["week_id"].map(week_map)
    return train_wide, val_wide


def make_loaders(train_wide, val_wide, batch_size: int, seed: int):
    factory = DataLoaderFactory(num_workers=4, pin_memory=True, persistent_workers=True)
    train_ds = MultiProductDataset(train_wide, n=n_upcs)
    val_ds = MultiProductDataset(val_wide, n=n_upcs)
    g = torch.Generator()
    g.manual_seed(seed)
    train_loader = factory.create_train_loader(
        train_ds, batch_size=batch_size, shuffle=True, drop_last=True, generator=g,
    )
    val_loader = factory.create_eval_loader(val_ds, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader


def config_from_params(params) -> MLPConfig:
    return replace(
        MLPConfig(),
        hidden=HIDDEN_OPTIONS[params["HIDDEN_KEY"]],
        dropout=float(params["DROPOUT"]),
        lr=float(params["LR"]),
        batch_size=int(params["BATCH_SIZE"]),
    )


def make_pipeline(cfg: MLPConfig, seed: int) -> DemandMLPPipeline:
    return DemandMLPPipeline(
        cfg,
        n=n_upcs,
        n_stores=n_stores,
        n_brands=n_brands,
        n_styles=n_styles,
        n_product_feats=n_product_feats,
        device=device,
        seed=seed,
    )


def fit_metrics(train_fold, val_fold, cfg: MLPConfig, seed: int) -> dict:
    """Search path: predictive metrics + ICDN elast_score (no row explosion)."""
    set_all_seeds(seed)
    train_wide, val_wide = encode_fold(train_fold, val_fold)
    train_loader, val_loader = make_loaders(train_wide, val_wide, cfg.batch_size, seed)
    pipe = make_pipeline(cfg, seed)
    pipe.fit(train_loader, val_loader)
    return {**pipe.metrics(val_loader), **pipe.elasticity_score(val_loader)}


def fit_evaluate(train_fold, val_fold, cfg: MLPConfig, seed: int):
    """Eval path: metrics + Jacobian elasticities."""
    set_all_seeds(seed)
    train_wide, val_wide = encode_fold(train_fold, val_fold)
    train_loader, val_loader = make_loaders(train_wide, val_wide, cfg.batch_size, seed)
    pipe = make_pipeline(cfg, seed)
    pipe.fit(train_loader, val_loader)
    return pipe.evaluate(
        val_loader, store_cats=store_cats, upc_names=upc_names, week_cats=week_cats,
    )

# Optuna

In [7]:
trial_records = []

def objective(trial, fold_splits, study_tag: str):
    params = {
        "HIDDEN_KEY": trial.suggest_categorical("HIDDEN_KEY", list(HIDDEN_OPTIONS)),
        "DROPOUT": trial.suggest_float("DROPOUT", 0.0, 0.3),
        "LR": trial.suggest_float("LR", 1e-4, 1e-2, log=True),
        "BATCH_SIZE": trial.suggest_categorical("BATCH_SIZE", [256, 512, 1024]),
    }
    print(f"\n{'='*70}\n[{study_tag}] Trial {trial.number}\n{params}\n{'='*70}")
    cfg = config_from_params(params)

    rows = []
    for fold_id, (tr, va) in enumerate(fold_splits):
        for seed in TUNE_SEEDS:
            m = fit_metrics(tr, va, cfg, seed)
            rows.append({"fold": fold_id, "seed": seed, **m})

    df_t = pd.DataFrame(rows)

    mean_r2 = float(df_t["r2_val"].mean())
    std_r2 = float(df_t["r2_val"].std(ddof=1)) if len(df_t) > 1 else 0.0
    mean_elast = float(df_t["elast_score"].mean())
    std_elast = float(df_t["elast_score"].std(ddof=1)) if len(df_t) > 1 else 0.0
    robust_r2 = mean_r2 - 0.25 * std_r2
    robust_elast = mean_elast - 0.25 * std_elast

    trial.set_user_attr("mean_r2", mean_r2)
    trial.set_user_attr("std_r2", std_r2)
    trial.set_user_attr("robust_r2", robust_r2)
    trial.set_user_attr("mean_elast_score", mean_elast)
    trial.set_user_attr("std_elast_score", std_elast)
    trial.set_user_attr("robust_elast", robust_elast)
    trial.set_user_attr("mean_mae", float(df_t["mae_val"].mean()))
    trial.set_user_attr("mean_rmse", float(df_t["rmse_val"].mean()))

    df_t["trial"] = trial.number
    df_t["study_tag"] = study_tag
    for k, v in params.items():
        df_t[k] = v
    trial_records.extend(df_t.to_dict("records"))
    print(
        f"[{study_tag}] Trial {trial.number} summary | "
        f"mean_R2={mean_r2:.4f} std_R2={std_r2:.4f} "
        f"robust_R2={robust_r2:.4f} | "
        f"mean_Elast_Score={mean_elast:.4f} std_Elast_Score={std_elast:.4f} "
        f"robust_Elast_Score={robust_elast:.4f} | "
        f"S_select={robust_r2 + robust_elast:.4f}"
    )
    return robust_r2, robust_elast


def study_to_summary(study) -> pd.DataFrame:
    df = pd.DataFrame([
        {
            "trial": t.number,
            "robust_r2": t.user_attrs.get("robust_r2", np.nan),
            "robust_elast": t.user_attrs.get("robust_elast", np.nan),
            "mean_r2": t.user_attrs.get("mean_r2", np.nan),
            "std_r2": t.user_attrs.get("std_r2", np.nan),
            "mean_elast_score": t.user_attrs.get("mean_elast_score", np.nan),
            "std_elast_score": t.user_attrs.get("std_elast_score", np.nan),
            "mean_mae": t.user_attrs.get("mean_mae", np.nan),
            "mean_rmse": t.user_attrs.get("mean_rmse", np.nan),
            **t.params,
        }
        for t in study.trials if t.values is not None
    ])
    df["robust_score"] = df["robust_r2"].fillna(0.0) + df["robust_elast"].fillna(0.0)
    return df.sort_values("robust_score", ascending=False)


def best_payload_mlp(best_row: pd.Series) -> dict:
    return {
        "trial": int(best_row["trial"]),
        "robust_score": float(best_row["robust_score"]),
        "mean_r2": float(best_row["mean_r2"]),
        "std_r2": float(best_row["std_r2"]),
        "mean_elast_score": float(best_row["mean_elast_score"]),
        "std_elast_score": float(best_row["std_elast_score"]),
        "params": {
            "HIDDEN_KEY": str(best_row["HIDDEN_KEY"]),
            "DROPOUT": float(best_row["DROPOUT"]),
            "LR": float(best_row["LR"]),
            "BATCH_SIZE": int(best_row["BATCH_SIZE"]),
        },
    }


def run_optuna(fold_splits, tag: str, n_trials: int):
    study = optuna.create_study(
        directions=["maximize", "maximize"],
        study_name=tag,
        storage=f"sqlite:///{NESTED_DIR / (tag + '.db')}",
        load_if_exists=True,
    )
    n_done = len([t for t in study.trials if t.values is not None])
    n_left = max(0, n_trials - n_done)
    print(f"{tag}: {n_done} done, {n_left} left")
    if n_left:
        study.optimize(
            lambda t, splits=fold_splits, tag=tag: objective(t, splits, tag),
            n_trials=n_left,
        )
    return study

In [8]:
outer_best = []

for plan in nested_plans:
    k = plan["outer_id"]
    tag = f"mlp_nested_outer{k}"
    study = run_optuna(plan["inner_splits"], tag, N_TRIALS_NESTED)
    summary = study_to_summary(study)
    payload = best_payload_mlp(summary.iloc[0])
    payload["outer_id"] = k
    payload["protocol"] = PROTOCOL
    with open(NESTED_DIR / f"mlp_outer{k}_best_params.json", "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)
    summary.to_csv(NESTED_DIR / f"mlp_outer{k}_trials.csv", index=False)
    outer_best.append(payload)
    print(f"Saved {tag} trial {payload['trial']} robust_score={payload['robust_score']:.4f}")

with open(NESTED_DIR / "mlp_outer_best_params.json", "w", encoding="utf-8") as f:
    json.dump(outer_best, f, indent=2, ensure_ascii=False)

[I 2026-09-03 10:16:33,299] A new study created in RDB with name: mlp_nested_outer0


mlp_nested_outer0: 0 done, 20 left

[mlp_nested_outer0] Trial 0
{'HIDDEN_KEY': '192_96', 'DROPOUT': 0.008131619736018213, 'LR': 0.00010247968737794945, 'BATCH_SIZE': 256}
early stop at epoch 221
early stop at epoch 209
early stop at epoch 203
early stop at epoch 235


[W 2026-09-03 10:17:51,848] Trial 0 failed with parameters: {'HIDDEN_KEY': '192_96', 'DROPOUT': 0.008131619736018213, 'LR': 0.00010247968737794945, 'BATCH_SIZE': 256} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/thebigmonster/Github/nn-elasticity/.venv/lib/python3.10/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "/tmp/ipykernel_16276/3587046064.py", line 102, in <lambda>
    lambda t, splits=fold_splits, tag=tag: objective(t, splits, tag),
  File "/tmp/ipykernel_16276/3587046064.py", line 16, in objective
    m = fit_metrics(tr, va, cfg, seed)
  File "/tmp/ipykernel_16276/2044747589.py", line 51, in fit_metrics
    pipe.fit(train_loader, val_loader)
  File "/home/thebigmonster/Github/nn-elasticity/src/benchmarks/demand_mlp.py", line 161, in fit
    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
  File "/home/thebigmonster/Github/nn-elasticity/.venv/lib/python3.10/sit

KeyboardInterrupt: 

In [ ]:
tag = "mlp_nested_holdout"
study_h = run_optuna(holdout_inner, tag, N_TRIALS_HOLDOUT)
holdout_best = best_payload_mlp(study_to_summary(study_h).iloc[0])
holdout_best["protocol"] = PROTOCOL
holdout_best["split"] = "holdout_80_20"
with open(NESTED_DIR / "mlp_holdout_best_params.json", "w", encoding="utf-8") as f:
    json.dump(holdout_best, f, indent=2, ensure_ascii=False)
print(json.dumps(holdout_best, indent=2))

# K-folds

In [ ]:
params_by_outer = {int(p["outer_id"]): p for p in outer_best}

fold_metrics_rows = []
fold_elasticity_rows = []

for plan in nested_plans:
    fold_id = plan["outer_id"]
    payload = params_by_outer[fold_id]
    cfg = config_from_params(payload["params"])
    print(f"=== Outer {fold_id} | trial {payload['trial']} | {cfg}")

    for seed in EVAL_SEEDS:
        print(f"=== Fold {fold_id} | Seed {seed} ===")
        metrics, elas = fit_evaluate(plan["outer_train"], plan["outer_val"], cfg, seed)
        fold_metrics_rows.append({
            "protocol": PROTOCOL,
            "run_type": "kfold_nested",
            "fold": fold_id,
            "seed": seed,
            "n_train": len(plan["outer_train"]),
            "n_val": len(plan["outer_val"]),
            "selected_trial": payload["trial"],
            **metrics,
        })
        elas = elas.copy()
        elas["protocol"] = PROTOCOL
        elas["run_type"] = "kfold_nested"
        elas["run_id"] = f"fold{fold_id}_seed{seed}"
        elas["fold"] = fold_id
        elas["seed"] = seed
        elas["bootstrap_run"] = np.nan
        fold_elasticity_rows.append(elas)
        print("  mae/rmse/r2", metrics["mae_val"], metrics["rmse_val"], metrics["r2_val"])

mlp_kfold_metrics_raw = pd.DataFrame(fold_metrics_rows)
mlp_kfold_elasticities_raw = pd.concat(fold_elasticity_rows, ignore_index=True)
print(mlp_kfold_metrics_raw)
print(mlp_kfold_elasticities_raw["type"].value_counts())

# Bootstrap

In [ ]:
bootstrap_sampler = BlockBootstrapSampler(
    week_col="week_id",
    block_size=BLOCK_SIZE,
    rng=np.random.default_rng(BASE_SEED),
)
BOOTSTRAP_SEED = EVAL_SEEDS[0]
cfg_h = config_from_params(holdout_best["params"])

bootstrap_metrics_rows = []
bootstrap_elasticity_rows = []

for b in range(N_BOOTSTRAP):
    print(f"=== Bootstrap {b + 1}/{N_BOOTSTRAP} ===")
    train_bs = bootstrap_sampler.sample(train_final, train_weeks_final)
    metrics, elas = fit_evaluate(train_bs, val_final, cfg_h, BOOTSTRAP_SEED)
    bootstrap_metrics_rows.append({
        "protocol": PROTOCOL,
        "run_type": "bootstrap_nested",
        "bootstrap_run": b,
        "seed": BOOTSTRAP_SEED,
        "n_train": len(train_bs),
        "n_val": len(val_final),
        "selected_trial": holdout_best["trial"],
        **metrics,
    })
    elas = elas.copy()
    elas["protocol"] = PROTOCOL
    elas["run_type"] = "bootstrap_nested"
    elas["run_id"] = f"bootstrap{b}"
    elas["fold"] = np.nan
    elas["seed"] = BOOTSTRAP_SEED
    elas["bootstrap_run"] = b
    bootstrap_elasticity_rows.append(elas)
    print("  mae/rmse/r2", metrics["mae_val"], metrics["rmse_val"], metrics["r2_val"])

mlp_bootstrap_metrics_raw = pd.DataFrame(bootstrap_metrics_rows)
mlp_bootstrap_elasticities_raw = pd.concat(bootstrap_elasticity_rows, ignore_index=True)
print(mlp_bootstrap_metrics_raw.head())

# Save

In [ ]:
mlp_kfold_metrics_raw.to_csv(DATA_DIR / "benchmark_mlp_kfold_metrics_raw_nested.csv", index=False)
mlp_kfold_elasticities_raw.to_csv(DATA_DIR / "benchmark_mlp_kfold_raw_nested.csv", index=False)
mlp_bootstrap_metrics_raw.to_csv(DATA_DIR / "benchmark_mlp_bootstrap_metrics_raw_nested.csv", index=False)
mlp_bootstrap_elasticities_raw.to_csv(DATA_DIR / "benchmark_mlp_bootstrap_raw_nested.csv", index=False)
print("Saved MLP nested k-fold + bootstrap")